# Do checks and quality flags for extracted data
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



In [2]:
#load data (model)
model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
nreports = 50
res_savename = f"llm_response_impact_labelled_reports_{model_name.replace('/', '_')}.csv"
response_df = pd.read_csv(DATA_OUT_LLMS+res_savename)


In [3]:
#load data (labelled)
fnla = "labelled_8reports_impact_laura.csv"
labelled_laura = pd.read_csv(DATA_LABELLED+fnla)
fnlu = "labelled_reports_impacts_luca.csv"
labelled_luca = pd.read_csv(DATA_LABELLED+fnlu)#.drop(["Unnamed: 0"],axis=1)



In [4]:
response_df

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,impactsAnnotation,impactValueMin,impactValueMax,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text,impactType
0,Affected People,43880.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",NaN,NaN,NaN,NaN,NaN,...,['TC PAM Global Relief Response : Across5 coun...,43880.0,43880.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
1,Affected People,34573.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",NaN,NaN,NaN,NaN,NaN,...,['TC PAM Global Recovery Response : Across5 co...,34573.0,34573.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
2,Residential Buildings,900.0,houses,exact,['Vanuatu'],['West Tanna'],NaN,NaN,NaN,NaN,...,['This is important feedback as the safe shelt...,900.0,900.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
3,Human Health and Wellbeing,85.0,per cent,exact,['Vanuatu'],NaN,NaN,NaN,NaN,NaN,...,"['In Vanuatu, the results of feedback mechanis...",85.0,85.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
4,"Access to Water, Sanitation, and Hygiene",138.0,households,exact,['Tuvalu'],NaN,NaN,NaN,NaN,NaN,...,['A total of13 rainwater harvesting systems ( ...,138.0,138.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272,Crop Production and Forestry,NaN,NaN,NaN,['Zambia'],NaN,2023.0,NaN,NaN,2024.0,...,"['projected production levels were minimal, an...",NaN,NaN,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN
273,Affected Livestock and Animals,NaN,NaN,NaN,['Zambia'],NaN,2023.0,NaN,NaN,2024.0,...,['almost half of surveyed households that kept...,NaN,NaN,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN
274,Other Economic and Livelihood Impacts,11.0,CHF million,exact,['Zambia'],NaN,2024.0,NaN,NaN,NaN,...,"['the IFRC, in support to the ZRCS, launched a...",11.0,11.0,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN
275,Access to Food,NaN,NaN,NaN,['Zambia'],NaN,NaN,NaN,NaN,NaN,...,['decreased access to water has also led to ou...,NaN,NaN,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN


In [5]:
#get rid of nans
response_df = response_df.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df.columns else response_df

In [6]:
response_df

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,impactsAnnotation,impactValueMin,impactValueMax,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text,impactType
0,Affected People,43880.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",NaN,NaN,NaN,NaN,NaN,...,['TC PAM Global Relief Response : Across5 coun...,43880.0,43880.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
1,Affected People,34573.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",NaN,NaN,NaN,NaN,NaN,...,['TC PAM Global Recovery Response : Across5 co...,34573.0,34573.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
2,Residential Buildings,900.0,houses,exact,['Vanuatu'],['West Tanna'],NaN,NaN,NaN,NaN,...,['This is important feedback as the safe shelt...,900.0,900.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
3,Human Health and Wellbeing,85.0,per cent,exact,['Vanuatu'],NaN,NaN,NaN,NaN,NaN,...,"['In Vanuatu, the results of feedback mechanis...",85.0,85.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
4,"Access to Water, Sanitation, and Hygiene",138.0,households,exact,['Tuvalu'],NaN,NaN,NaN,NaN,NaN,...,['A total of13 rainwater harvesting systems ( ...,138.0,138.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272,Crop Production and Forestry,NaN,NaN,NaN,['Zambia'],NaN,2023.0,NaN,NaN,2024.0,...,"['projected production levels were minimal, an...",NaN,NaN,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN
273,Affected Livestock and Animals,NaN,NaN,NaN,['Zambia'],NaN,2023.0,NaN,NaN,2024.0,...,['almost half of surveyed households that kept...,NaN,NaN,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN
274,Other Economic and Livelihood Impacts,11.0,CHF million,exact,['Zambia'],NaN,2024.0,NaN,NaN,NaN,...,"['the IFRC, in support to the ZRCS, launched a...",11.0,11.0,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN
275,Access to Food,NaN,NaN,NaN,['Zambia'],NaN,NaN,NaN,NaN,NaN,...,['decreased access to water has also led to ou...,NaN,NaN,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN


In [7]:
#convert numerical columns
num_cols = ["impactValue"]#"startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"
list_cols = ["location", "hazards", "impactsAnnotation"]
response_df_proc = cp.deepcopy(response_df)
response_df_proc = format_output(response_df_proc, num_cols=num_cols, list_cols=list_cols)



In [8]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [9]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Sanity checks
1. ValueInText: ImpactValue should be in the original text
2. MaxPop: ImpactValue with “people” unit should be smaller than country’s population
3. OrigSentence: Annotation sentence should be present in the original text
4. UnknownImpType: Inferred impactType must be in the allowed list
5. UnknownHaz: Inferred hazardType must be in allowed list
6. Location partially undefined. 


In [10]:
response_df_proc.impactUnit.value_counts()

impactUnit
people                              117
houses                               24
households                           11
families                              5
per cent                              4
hectares                              3
livestock                             3
latrines                              3
schools                               3
hectares of crops                     2
person                                2
PHP                                   2
billion Yuan                          2
buildings                             2
health facilities                     2
homes                                 2
% of area                             1
landslides                            1
% of infrastructure                   1
hectares of land                      1
% of GDP                              1
health units                          1
years                                 1
million USD                           1
workplaces                   

In [11]:
response_df_exploded = explode_lists(response_df_proc)



/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:89: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


In [12]:
response_df_exploded.hazards.value_counts()

hazards
Flood                       179
Tropical storm              109
Extreme cold temperature     80
Wildfire                     72
Other Storm                  46
Drought                      41
Volcanic activity            25
Mass movement                15
Earthquake                   14
Tsunami                      14
Epidemic                     13
Name: count, dtype: int64

In [13]:
#value in original text
response_df_proc = flag_value_in_text(response_df_proc)

In [14]:
#value in original text
response_df_proc = flag_value_in_text(response_df_proc)

In [15]:
response_df_proc.value_in_text.value_counts()

value_in_text
False    115
True      95
Name: count, dtype: int64

In [16]:
response_df_proc[response_df_proc["value_in_text"] == False]

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text,impactType,country_iso3,country_iso3_kw,value_in_text
0,Affected People,43880.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",[],NaN,NaN,NaN,NaN,...,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,None,FJI,False
1,Affected People,34573.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",[],NaN,NaN,NaN,NaN,...,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,None,FJI,False
8,Human Deaths,1197.0,people,exact,['Afghanistan'],"[Badakhshan, Badghis, Balkh, Farah, Faryab, Gh...",2024.0,3,3.0,NaN,...,MDRAF014,Afghanistan,2024-12-31 00:00:00,https://go-api.ifrc.org/api/downloadfile/87310...,Cold Wave,['Page 1 / 19Description of the Event Map of p...,NaN,None,AFG,False
9,Injured People,2217.0,people,exact,['Afghanistan'],"[Badakhshan, Badghis, Balkh, Farah, Faryab, Gh...",2024.0,3,3.0,NaN,...,MDRAF014,Afghanistan,2024-12-31 00:00:00,https://go-api.ifrc.org/api/downloadfile/87310...,Cold Wave,['Page 1 / 19Description of the Event Map of p...,NaN,None,AFG,False
10,Affected People,325205.0,people,exact,['Afghanistan'],"[Badakhshan, Badghis, Balkh, Farah, Faryab, Gh...",2024.0,3,3.0,NaN,...,MDRAF014,Afghanistan,2024-12-31 00:00:00,https://go-api.ifrc.org/api/downloadfile/87310...,Cold Wave,['Page 1 / 19Description of the Event Map of p...,NaN,None,AFG,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,Informal settlements,17000.0,people,exact,['Yemen'],[IDP sites],2022.0,5,NaN,2022.0,...,MDRYE011,Yemen,2023-05-29 00:00:00,https://go-api.ifrc.org/api/DownloadFile/67143...,Flood,"[""SITUATION ANALYSIS Description of the disast...",NaN,None,YEM,False
267,Affected People,6000000.0,people,exact,['Zambia'],"[Lusaka, Luapula, Western, Eastern, Southern, ...",2024.0,2,29.0,NaN,...,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,None,ZMB,False
268,Affected People,2000000.0,people,exact,['Zambia'],[],2023.0,10,NaN,2024.0,...,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,None,ZMB,False
269,Affected People,58000.0,people,exact,['Zambia'],[],2023.0,10,NaN,2024.0,...,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,None,ZMB,False


In [17]:
response_df_proc[(response_df_proc.country_iso3_kw.isna() | response_df_proc.country_iso3_kw.isnull())]

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text,impactType,country_iso3,country_iso3_kw,value_in_text
245,Human Deaths,53000.0,people,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False
246,Injured People,107000.0,people,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False
247,Displaced People,3000000.0,people,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False
248,Homeless People,262000.0,buildings,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False
249,Affected People,15700000.0,people,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False
250,Residential Buildings,300000.0,buildings,NaN,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False
251,Other Economic and Livelihood Impacts,79.0,million USD,exact,['Türkiye'],[Southeastern Türkiye],2021.0,NaN,NaN,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,True
252,Affected Livestock and Animals,220000.0,workplaces,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False
253,Human Health and Wellbeing,NaN,NaN,NaN,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,NaN
254,Access to Healthcare,NaN,NaN,NaN,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,MDRTR004,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,NaN


In [18]:
#impacted people must be less than population
country_pop = pd.read_csv(DATA_PATH +"API_SP.POP.TOTL_DS2_en_csv_v2_131993/"+"API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv",sep=',', header=2)
country_pop = country_pop.dropna(how="all",axis=1)

response_df_proc = pop_cntry_check(response_df_proc, country_pop)

In [19]:
response_df_proc.pop_cntry_check.value_counts()

pop_cntry_check
True     112
False      4
Name: count, dtype: int64

In [20]:
response_df_proc[response_df_proc["pop_cntry_check"] == False]

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,country_kw,reportDate,reportLink,disasterType,nathaz_text,impactType,country_iso3,country_iso3_kw,value_in_text,pop_cntry_check
245,Human Deaths,53000.0,people,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False,False
246,Injured People,107000.0,people,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False,False
247,Displaced People,3000000.0,people,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False,False
249,Affected People,15700000.0,people,exact,['Türkiye'],[Southeastern Türkiye],2023.0,2,6.0,NaN,...,Turkey,2024-12-06 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Earthquake,['SITUATION ANALYSIS Description of the crisis...,NaN,None,None,False,False


In [21]:
# check if hazard are in list
hazard_list = hazard_main_types_emdat_extended
response_df_proc = flag_hazard(response_df_proc, hazard_list)
response_df_proc.unknown_haz.value_counts()

unknown_haz
False    268
True       7
Name: count, dtype: int64

In [22]:
response_df_proc[response_df_proc["unknown_haz"] == True]

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,reportDate,reportLink,disasterType,nathaz_text,impactType,country_iso3,country_iso3_kw,value_in_text,pop_cntry_check,unknown_haz
129,Human Deaths,4140.0,people,exact,['Indonesia'],[Central Sulawesi],2018.0,9,28.0,NaN,...,2021-09-20 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=44...,Earthquake,['The final report will reflect the overall co...,NaN,None,IDN,False,True,True
130,Injured People,7100.0,people,exact,['Indonesia'],[Central Sulawesi],2018.0,9,28.0,NaN,...,2021-09-20 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=44...,Earthquake,['The final report will reflect the overall co...,NaN,None,IDN,False,True,True
131,Displaced People,173000.0,people,exact,['Indonesia'],[Central Sulawesi],2018.0,9,28.0,NaN,...,2021-09-20 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=44...,Earthquake,['The final report will reflect the overall co...,NaN,None,IDN,False,True,True
132,Missing People,705.0,people,exact,['Indonesia'],[Central Sulawesi],2018.0,9,28.0,NaN,...,2021-09-20 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=44...,Earthquake,['The final report will reflect the overall co...,NaN,None,IDN,True,True,True
133,Residential Buildings,110000.0,houses,exact,['Indonesia'],[Central Sulawesi],2018.0,9,28.0,NaN,...,2021-09-20 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=44...,Earthquake,['The final report will reflect the overall co...,NaN,None,IDN,False,NaN,True
134,Healthcare Infrastructure,320.0,health facilities,exact,['Indonesia'],[Central Sulawesi],2018.0,9,28.0,NaN,...,2021-09-20 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=44...,Earthquake,['The final report will reflect the overall co...,NaN,None,IDN,True,NaN,True
135,Education Infrastructure,1300.0,schools,exact,['Indonesia'],[Central Sulawesi],2018.0,9,28.0,NaN,...,2021-09-20 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=44...,Earthquake,['The final report will reflect the overall co...,NaN,None,IDN,False,NaN,True


In [23]:
# check if impact are in list
def flag_impactSubtype(extracted_data, impact_list):
    def check_imp(x):
        return x["impactSubtype"] not in impact_list
    extracted_data["unknown_impactSubtype"] = np.nan
    extracted_data["unknown_impactSubtype"] = extracted_data.apply(check_imp, axis=1)
    return extracted_data

impact_list = impactSubtype_list
response_df_proc = flag_impactSubtype(response_df_proc, impact_list)
response_df_proc.unknown_impactSubtype.value_counts()

unknown_impactSubtype
False    274
True       1
Name: count, dtype: int64

In [33]:
# save
savename = "flaged_" + res_savename
response_df_proc.to_csv(DATA_OUT_LLMS + savename, index=False)